# Normalize data: REDS Index and REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from hk1_r12ter_analysis.config import INTERIM_DATA_DIR, logger
from hk1_r12ter_analysis.data import sample_normalize_feature_scale_data
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data and normalize

In [ ]:
normalization_inputs = (
    # Sample Normalization, Data Transformation, and Data Scaling
    [None, None, None],
    ["median", None, "auto"],
    ["median", "log2", "pareto"],  # For limma package
)

# For scaling/normalization factor(s), or normalization by a reference sample/feature
reference = None
#  For log transformations
add_constant = None
# DataFrame format axis=0: (Samples, Features) or axis=1: (Features, Samples)
axis = 0

# Input data arguments
input_dirpath = INTERIM_DATA_DIR / "REDS" / "Filtered"
output_dirpath = INTERIM_DATA_DIR / "REDS" / "Normalized"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Load REDS Index data

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"
data_types = [
    "Metabolomics",
    "Proteomics",
]

### Normalize

In [ ]:
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for data_type in data_types:
    # Load data
    filename_args = (source_type, "Donor", "RBC", data_type)
    logger.debug(f"Processing data for '{'-'.join(filename_args)}'...")
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
    for norm_str in normalization_strs:
        norm_method, transform_method, scaling_method = norm_str.split("-")
        output_dirpath_norm = output_dirpath / norm_str
        output_dirpath_norm.mkdir(parents=True, exist_ok=True)
        logger.debug(
            f"Processing data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}..."
        )
        # Exception for REDS-Index Metabolomics which are already sample normalized by median
        if source_type == "REDS-Index" and data_type == "Metabolomics":
            df_norm = sample_normalize_feature_scale_data(
                df,
                normalization_method=None,
                transformation_method=transform_method,
                scaling_method=scaling_method,
                axis=axis,
                reference=reference,  # TODO handle multiple provided values
                add_constant=add_constant,  # TODO handle multiple provided values
            )

        else:
            df_norm = sample_normalize_feature_scale_data(
                df,
                normalization_method=norm_method,
                transformation_method=transform_method,
                scaling_method=scaling_method,
                axis=axis,
                reference=reference,  # TODO handle multiple provided values
                add_constant=add_constant,  # TODO handle multiple provided values
            )
        filename = make_filename(*filename_args)
        save_dataframe_to_file(df_norm, output_dirpath_norm / filename, index=True)
        logger.info(
            f"Processed data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}."
        )
    logger.info(f"Processed data for '{'-'.join(filename_args)}'.")

### Load REDS Recall data

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
data_types = [
    "Proteomics",
    "Metabolomics",
    "Lipidomics",
    # "Lipidomics-DegUnsat",
    # "Lipidomics-LipidClass",
    "TraceElements",
    # "Vesicles-mtDNA",
]

### Normalize

In [ ]:
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for data_type in data_types:
    # Load data
    filename_args = (source_type, "Donor", "RBC", data_type)
    logger.debug(f"Processing data for '{'-'.join(filename_args)}'...")
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
    for norm_str in normalization_strs:
        norm_method, transform_method, scaling_method = norm_str.split("-")
        output_dirpath_norm = output_dirpath / norm_str
        output_dirpath_norm.mkdir(parents=True, exist_ok=True)
        logger.debug(
            f"Processing data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}..."
        )
        # Exception for REDS-Recall Lipidomics which are already normalized by controls
        if source_type == "REDS-Recall" and data_type == "Lipidomics":
            df_norm = sample_normalize_feature_scale_data(
                df,
                normalization_method=None,
                transformation_method=transform_method,
                scaling_method=scaling_method,
                axis=axis,
                reference=reference,  # TODO handle multiple provided values
                add_constant=add_constant,  # TODO handle multiple provided values
            )
        else:
            df_norm = sample_normalize_feature_scale_data(
                df,
                normalization_method=norm_method,
                transformation_method=transform_method,
                scaling_method=scaling_method,
                axis=axis,
                reference=reference,  # TODO handle multiple provided values
                add_constant=add_constant,  # TODO handle multiple provided values
            )
        filename = make_filename(*filename_args)
        save_dataframe_to_file(df_norm, output_dirpath_norm / filename, index=True)
        logger.info(
            f"Processed data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}."
        )
    logger.info(f"Processed data for '{'-'.join(filename_args)}'.")